In [3]:
# =============================================================================
# 08_download_sentinel1.ipynb
#
# Purpose:
# Download Sentinel-1 SAR image chips for the same matched treatment and
# counterfactual sites used in the Sentinel-2 workflow.
#
# Inputs:
#
# datasets/finals/treatment_sites.geojson
# datasets/finals/counterfactual_sites.geojson
#
# Outputs:
#
# datasets/finals/sentinel1/
# ├── treatment/
# │   ├── before/
# │   └── after/
# ├── counterfactual/
# │   ├── before/
# │   └── after/
# └── previews/
#
# Exported GeoTIFF band order:
#
# Band 1: VV
# Band 2: VH
# Band 3: VV_minus_VH
#
# COPERNICUS/S1_GRD values are in decibels.
# VV_minus_VH is therefore the difference between VV and VH in dB.
# =============================================================================


# =============================================================================
# 1. Load packages
# =============================================================================

from pathlib import Path
import json
import time

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from tqdm.auto import tqdm

print("Packages loaded successfully.")


# =============================================================================
# 2. Define project paths
# =============================================================================

BASE_DIR = Path(
    "/Users/gaoyujuan/REAP Dropbox/Gao yujuan/"
    "Virginia Tech/CALS/datasets"
)

FINALS_DIR = BASE_DIR / "finals"
METADATA_DIR = BASE_DIR / "metadata"

TREATMENT_SITES_FILE = (
    FINALS_DIR /
    "treatment_sites.geojson"
)

COUNTERFACTUAL_SITES_FILE = (
    FINALS_DIR /
    "counterfactual_sites.geojson"
)

SENTINEL1_DIR = (
    FINALS_DIR /
    "sentinel1"
)

S1_TREATMENT_DIR = (
    SENTINEL1_DIR /
    "treatment"
)

S1_COUNTERFACTUAL_DIR = (
    SENTINEL1_DIR /
    "counterfactual"
)

S1_TREATMENT_BEFORE_DIR = (
    S1_TREATMENT_DIR /
    "before"
)

S1_TREATMENT_AFTER_DIR = (
    S1_TREATMENT_DIR /
    "after"
)

S1_COUNTERFACTUAL_BEFORE_DIR = (
    S1_COUNTERFACTUAL_DIR /
    "before"
)

S1_COUNTERFACTUAL_AFTER_DIR = (
    S1_COUNTERFACTUAL_DIR /
    "after"
)

S1_PREVIEW_DIR = (
    SENTINEL1_DIR /
    "previews"
)

for folder in [
    FINALS_DIR,
    METADATA_DIR,
    SENTINEL1_DIR,
    S1_TREATMENT_DIR,
    S1_COUNTERFACTUAL_DIR,
    S1_TREATMENT_BEFORE_DIR,
    S1_TREATMENT_AFTER_DIR,
    S1_COUNTERFACTUAL_BEFORE_DIR,
    S1_COUNTERFACTUAL_AFTER_DIR,
    S1_PREVIEW_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Sentinel-1 output directory:")
print(SENTINEL1_DIR)


# =============================================================================
# 3. Check required input files
# =============================================================================

required_files = [
    TREATMENT_SITES_FILE,
    COUNTERFACTUAL_SITES_FILE,
]

missing_files = [
    file
    for file in required_files
    if not file.exists()
]

if missing_files:

    raise FileNotFoundError(
        "One or more site files are missing.\n"
        "Run 06_create_sites.ipynb first.\n\n"
        "Missing files:\n"
        + "\n".join(
            str(file)
            for file in missing_files
        )
    )

print("Required site files found.")


# =============================================================================
# 4. Authenticate and initialize Google Earth Engine
# =============================================================================

EARTH_ENGINE_PROJECT = (
    "hurricane-504721"
)

try:

    ee.Initialize(
        project=EARTH_ENGINE_PROJECT
    )

    print(
        "Earth Engine initialized using "
        "existing credentials."
    )

except Exception as initialization_error:

    print(
        "Earth Engine initialization failed."
    )

    print(
        initialization_error
    )

    print(
        "\nStarting Earth Engine authentication..."
    )

    ee.Authenticate()

    ee.Initialize(
        project=EARTH_ENGINE_PROJECT
    )

    print(
        "Earth Engine authenticated and initialized."
    )


# =============================================================================
# 5. Define before and after periods
#
# The periods match the Sentinel-2 workflow.
# =============================================================================

IMAGE_PERIODS = {
    "before": {
        "start": "2024-08-15",
        "end": "2024-09-23",
    },
    "after": {
        "start": "2024-10-01",
        "end": "2024-10-31",
    },
}

print("Image periods:")

for period, dates in IMAGE_PERIODS.items():

    print(
        period,
        dates["start"],
        dates["end"],
    )


# =============================================================================
# 6. Define Sentinel-1 settings
# =============================================================================

SENTINEL1_COLLECTION = (
    "COPERNICUS/S1_GRD"
)

# Keep one orbit direction for more consistent before-after comparisons.
#
# Begin with DESCENDING. The diagnostic section below reports whether
# sufficient scenes are available. Change to ASCENDING if needed.
ORBIT_PASS = "ASCENDING"

INSTRUMENT_MODE = "IW"

EXPORT_SCALE_METERS = 10

EXPORT_CRS = "EPSG:32617"

OVERWRITE_EXISTING = False

CREATE_PREVIEWS = True

TEST_DOWNLOAD_FIRST = True

RETRY_ATTEMPTS = 3

RETRY_WAIT_SECONDS = 10

BAND_NAMES = [
    "VV",
    "VH",
    "VV_minus_VH",
]


# =============================================================================
# 7. Load treatment and counterfactual site polygons
# =============================================================================

treatment_sites = gpd.read_file(
    TREATMENT_SITES_FILE
)

counterfactual_sites = gpd.read_file(
    COUNTERFACTUAL_SITES_FILE
)


def standardize_site_layer(
    site_layer,
    expected_group,
):
    """
    Standardize a treatment or counterfactual site layer.
    """

    output = site_layer.copy()

    if output.crs is None:

        output = output.set_crs(
            "EPSG:4326"
        )

    else:

        output = output.to_crs(
            "EPSG:4326"
        )

    required_columns = [
        "site_id",
        "pair_id",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in output.columns
    ]

    if missing_columns:

        raise ValueError(
            f"The {expected_group} site file is missing "
            f"required columns: {missing_columns}"
        )

    output[
        "group"
    ] = expected_group

    output = output.loc[
        output.geometry.notna()
    ].copy()

    output = output.loc[
        ~output.geometry.is_empty
    ].copy()

    output = output.reset_index(
        drop=True
    )

    return output


treatment_sites = standardize_site_layer(
    treatment_sites,
    "treatment",
)

counterfactual_sites = standardize_site_layer(
    counterfactual_sites,
    "counterfactual",
)

print(
    "Treatment sites:",
    len(treatment_sites),
)

print(
    "Counterfactual sites:",
    len(counterfactual_sites),
)


# =============================================================================
# 8. Verify pair alignment
# =============================================================================

treatment_pairs = set(
    treatment_sites[
        "pair_id"
    ]
)

counterfactual_pairs = set(
    counterfactual_sites[
        "pair_id"
    ]
)

missing_counterfactual_pairs = (
    treatment_pairs -
    counterfactual_pairs
)

missing_treatment_pairs = (
    counterfactual_pairs -
    treatment_pairs
)

if missing_counterfactual_pairs:

    print(
        "Warning: treatment pairs without counterfactual sites:"
    )

    print(
        sorted(
            missing_counterfactual_pairs
        )
    )

if missing_treatment_pairs:

    print(
        "Warning: counterfactual pairs without treatment sites:"
    )

    print(
        sorted(
            missing_treatment_pairs
        )
    )


# =============================================================================
# 9. Combine all site layers
# =============================================================================

all_sites = gpd.GeoDataFrame(
    pd.concat(
        [
            treatment_sites,
            counterfactual_sites,
        ],
        ignore_index=True,
    ),
    crs="EPSG:4326",
)

all_sites = (
    all_sites
    .sort_values(
        [
            "pair_id",
            "group",
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Total sites:",
    len(all_sites),
)


# =============================================================================
# 10. Convert a Shapely geometry to Earth Engine geometry
# =============================================================================

def shapely_to_ee_geometry(
    geometry,
):
    """
    Convert one Shapely geometry to an Earth Engine geometry.
    """

    geometry_json = json.loads(
        gpd.GeoSeries(
            [geometry],
            crs="EPSG:4326",
        ).to_json()
    )

    feature = (
        geometry_json[
            "features"
        ][0]
    )

    return ee.Geometry(
        feature[
            "geometry"
        ]
    )


# =============================================================================
# 11. Create one combined geometry for availability diagnostics
# =============================================================================

combined_geometry = (
    all_sites
    .geometry
    .union_all()
)

combined_ee_geometry = shapely_to_ee_geometry(
    combined_geometry
)


# =============================================================================
# 12. Build the base Sentinel-1 collection
#
# Filters:
#
# - Interferometric Wide Swath mode
# - 10-meter resolution
# - VV polarization
# - VH polarization
# - selected orbit pass
# =============================================================================

def get_sentinel1_collection(
    region,
    start_date,
    end_date,
    orbit_pass,
):
    """
    Return a filtered dual-polarization Sentinel-1 collection.
    """

    collection = (
        ee.ImageCollection(
            SENTINEL1_COLLECTION
        )
        .filterBounds(
            region
        )
        .filterDate(
            start_date,
            end_date,
        )
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                INSTRUMENT_MODE,
            )
        )
        .filter(
            ee.Filter.eq(
                "resolution_meters",
                10,
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV",
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH",
            )
        )
        .filter(
            ee.Filter.eq(
                "orbitProperties_pass",
                orbit_pass,
            )
        )
    )

    return collection


# =============================================================================
# 13. Diagnose ascending and descending scene availability
# =============================================================================

orbit_diagnostic_records = []

for orbit_pass in [
    "ASCENDING",
    "DESCENDING",
]:

    for period, dates in IMAGE_PERIODS.items():

        collection = get_sentinel1_collection(
            region=combined_ee_geometry,
            start_date=dates["start"],
            end_date=dates["end"],
            orbit_pass=orbit_pass,
        )

        scene_count = (
            collection
            .size()
            .getInfo()
        )

        orbit_diagnostic_records.append(
            {
                "orbit_pass": orbit_pass,
                "period": period,
                "start_date": dates["start"],
                "end_date": dates["end"],
                "scene_count": scene_count,
            }
        )

orbit_diagnostics = pd.DataFrame(
    orbit_diagnostic_records
)

print("\nSentinel-1 orbit diagnostics:")
print(orbit_diagnostics)

ORBIT_DIAGNOSTICS_FILE = (
    METADATA_DIR /
    "sentinel1_orbit_diagnostics.csv"
)

orbit_diagnostics.to_csv(
    ORBIT_DIAGNOSTICS_FILE,
    index=False,
)

print("\nSelected orbit pass:")
print(ORBIT_PASS)


# =============================================================================
# 14. Check that the selected orbit has scenes in both periods
# =============================================================================

selected_orbit_counts = (
    orbit_diagnostics.loc[
        orbit_diagnostics[
            "orbit_pass"
        ] == ORBIT_PASS
    ]
)

if (
    selected_orbit_counts[
        "scene_count"
    ].min()
    == 0
):

    raise RuntimeError(
        f"The selected orbit pass, {ORBIT_PASS}, does not have "
        "Sentinel-1 scenes in both periods.\n\n"
        "Review the orbit diagnostic table and change ORBIT_PASS "
        "to the orbit with coverage in both periods."
    )


# =============================================================================
# 15. Inspect Sentinel-1 bands and properties
# =============================================================================

diagnostic_collection = get_sentinel1_collection(
    region=combined_ee_geometry,
    start_date=IMAGE_PERIODS["before"]["start"],
    end_date=IMAGE_PERIODS["before"]["end"],
    orbit_pass=ORBIT_PASS,
)

diagnostic_image = ee.Image(
    diagnostic_collection.first()
)

print("\nAvailable Sentinel-1 bands:")

print(
    diagnostic_image
    .bandNames()
    .getInfo()
)

print("\nExample Sentinel-1 image properties:")

print(
    diagnostic_image
    .toDictionary(
        [
            "instrumentMode",
            "orbitProperties_pass",
            "relativeOrbitNumber_start",
            "transmitterReceiverPolarisation",
            "resolution_meters",
        ]
    )
    .getInfo()
)


# =============================================================================
# 16. Mask unreliable Sentinel-1 edge pixels
#
# Very low backscatter values often occur along scene edges.
# This keeps pixels for which both VV and VH are greater than -35 dB.
# =============================================================================

def mask_sentinel1_edges(
    image,
):
    """
    Mask very low edge pixels and retain VV and VH bands.
    """

    vv = image.select(
        "VV"
    )

    vh = image.select(
        "VH"
    )

    valid_mask = (
        vv.gt(-35)
        .And(
            vh.gt(-35)
        )
    )

    return (
        image
        .select(
            [
                "VV",
                "VH",
            ]
        )
        .updateMask(
            valid_mask
        )
        .copyProperties(
            image,
            image.propertyNames(),
        )
    )


# =============================================================================
# 17. Build one Sentinel-1 composite
#
# The source collection is already in dB.
#
# VV_minus_VH:
#   VV dB minus VH dB
#
# A median composite is used to reduce speckle and short-term noise.
# =============================================================================

def build_sentinel1_composite(
    region,
    start_date,
    end_date,
):
    """
    Create a median Sentinel-1 VV/VH composite for one site and period.
    """

    collection = get_sentinel1_collection(
        region=region,
        start_date=start_date,
        end_date=end_date,
        orbit_pass=ORBIT_PASS,
    )

    scene_count = (
        collection
        .size()
        .getInfo()
    )

    if scene_count == 0:

        return None, 0, []

    relative_orbits = (
        collection
        .aggregate_array(
            "relativeOrbitNumber_start"
        )
        .distinct()
        .sort()
        .getInfo()
    )

    masked_collection = (
        collection.map(
            mask_sentinel1_edges
        )
    )

    composite = (
        masked_collection
        .median()
        .clip(
            region
        )
        .toFloat()
    )

    vv_minus_vh = (
        composite
        .select(
            "VV"
        )
        .subtract(
            composite.select(
                "VH"
            )
        )
        .rename(
            "VV_minus_VH"
        )
    )

    final_image = (
        composite
        .addBands(
            vv_minus_vh
        )
        .select(
            BAND_NAMES
        )
        .toFloat()
    )

    return (
        final_image,
        scene_count,
        relative_orbits,
    )


# =============================================================================
# 18. Determine output directory
# =============================================================================

def get_output_directory(
    group,
    period,
):
    """
    Return the Sentinel-1 folder for one group and period.
    """

    folder_map = {
        (
            "treatment",
            "before",
        ): S1_TREATMENT_BEFORE_DIR,

        (
            "treatment",
            "after",
        ): S1_TREATMENT_AFTER_DIR,

        (
            "counterfactual",
            "before",
        ): S1_COUNTERFACTUAL_BEFORE_DIR,

        (
            "counterfactual",
            "after",
        ): S1_COUNTERFACTUAL_AFTER_DIR,
    }

    key = (
        group,
        period,
    )

    if key not in folder_map:

        raise ValueError(
            f"Unsupported group-period combination: {key}"
        )

    return folder_map[
        key
    ]


# =============================================================================
# 19. Create Sentinel-1 preview
#
# Preview channels:
#
# Red   = VV
# Green = VH
# Blue  = VV minus VH
#
# This is a false-color radar preview, not a natural-color image.
# =============================================================================

def percentile_stretch(
    array,
    lower_percentile=2,
    upper_percentile=98,
):
    """
    Percentile-stretch one raster array to values between 0 and 1.
    """

    array = array.astype(
        "float32"
    )

    valid = array[
        np.isfinite(
            array
        )
    ]

    if valid.size == 0:

        return np.zeros_like(
            array,
            dtype="float32",
        )

    lower = np.nanpercentile(
        valid,
        lower_percentile,
    )

    upper = np.nanpercentile(
        valid,
        upper_percentile,
    )

    if upper <= lower:

        upper = lower + 1e-6

    stretched = np.clip(
        (
            array -
            lower
        )
        /
        (
            upper -
            lower
        ),
        0,
        1,
    )

    stretched[
        ~np.isfinite(
            stretched
        )
    ] = 0

    return stretched


def create_sentinel1_preview(
    tiff_file,
    preview_file,
):
    """
    Create a false-color Sentinel-1 PNG preview.
    """

    tiff_file = Path(
        tiff_file
    )

    preview_file = Path(
        preview_file
    )

    with rasterio.open(
        tiff_file
    ) as source:

        vv = source.read(
            1,
            masked=True,
        ).filled(
            np.nan
        )

        vh = source.read(
            2,
            masked=True,
        ).filled(
            np.nan
        )

        difference = source.read(
            3,
            masked=True,
        ).filled(
            np.nan
        )

    preview = np.stack(
        [
            percentile_stretch(
                vv
            ),
            percentile_stretch(
                vh
            ),
            percentile_stretch(
                difference
            ),
        ],
        axis=-1,
    )

    preview_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.figure(
        figsize=(6, 6)
    )

    plt.imshow(
        preview
    )

    plt.axis(
        "off"
    )

    plt.tight_layout(
        pad=0
    )

    plt.savefig(
        preview_file,
        dpi=200,
        bbox_inches="tight",
        pad_inches=0,
    )

    plt.close()

    return True


# =============================================================================
# 20. Download one Sentinel-1 site-period image
# =============================================================================

def download_site_image(
    site_row,
    period,
):
    """
    Build and download one Sentinel-1 image chip.
    """

    site_id = str(
        site_row[
            "site_id"
        ]
    )

    pair_id = str(
        site_row[
            "pair_id"
        ]
    )

    group = str(
        site_row[
            "group"
        ]
    )

    start_date = (
        IMAGE_PERIODS[
            period
        ][
            "start"
        ]
    )

    end_date = (
        IMAGE_PERIODS[
            period
        ][
            "end"
        ]
    )

    output_directory = get_output_directory(
        group,
        period,
    )

    output_file = (
        output_directory /
        f"{site_id}_{period}_sentinel1.tif"
    )

    preview_file = (
        S1_PREVIEW_DIR /
        group /
        period /
        f"{site_id}_{period}_sentinel1_preview.png"
    )

    record = {
        "site_id": site_id,
        "pair_id": pair_id,
        "group": group,
        "period": period,
        "start_date": start_date,
        "end_date": end_date,
        "sensor": "Sentinel-1 SAR GRD",
        "collection": SENTINEL1_COLLECTION,
        "instrument_mode": INSTRUMENT_MODE,
        "orbit_pass": ORBIT_PASS,
        "resolution_m": EXPORT_SCALE_METERS,
        "export_crs": EXPORT_CRS,
        "bands": ",".join(
            BAND_NAMES
        ),
        "scene_count": np.nan,
        "relative_orbits": None,
        "status": None,
        "image_path": str(
            output_file
        ),
        "preview_path": str(
            preview_file
        ),
        "file_size_bytes": np.nan,
        "error": None,
    }

    if (
        output_file.exists()
        and not OVERWRITE_EXISTING
    ):

        record[
            "status"
        ] = "existing"

        record[
            "file_size_bytes"
        ] = output_file.stat().st_size

        if (
            CREATE_PREVIEWS
            and not preview_file.exists()
        ):

            create_sentinel1_preview(
                output_file,
                preview_file,
            )

        return record

    try:

        region = shapely_to_ee_geometry(
            site_row.geometry
        )

        (
            image,
            scene_count,
            relative_orbits,
        ) = build_sentinel1_composite(
            region=region,
            start_date=start_date,
            end_date=end_date,
        )

        record[
            "scene_count"
        ] = scene_count

        record[
            "relative_orbits"
        ] = ",".join(
            str(value)
            for value in relative_orbits
        )

        if image is None:

            record[
                "status"
            ] = "no_scenes"

            record[
                "error"
            ] = (
                "No dual-polarization Sentinel-1 scenes "
                "were found for this site and period."
            )

            return record

        output_file.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        last_error = None

        for attempt in range(
            1,
            RETRY_ATTEMPTS + 1,
        ):

            try:

                print(
                    f"\nDownloading {site_id} {period}; "
                    f"attempt {attempt}..."
                )

                geemap.ee_export_image(
                    image,
                    filename=str(
                        output_file
                    ),
                    scale=EXPORT_SCALE_METERS,
                    region=region,
                    crs=EXPORT_CRS,
                    file_per_band=False,
                )

                if output_file.exists():

                    break

                raise FileNotFoundError(
                    "geemap finished without creating "
                    "the expected Sentinel-1 GeoTIFF."
                )

            except Exception as download_error:

                last_error = download_error

                print(
                    f"Attempt {attempt} failed for "
                    f"{site_id} {period}: "
                    f"{download_error}"
                )

                if attempt < RETRY_ATTEMPTS:

                    time.sleep(
                        RETRY_WAIT_SECONDS
                    )

        if not output_file.exists():

            raise RuntimeError(
                f"Sentinel-1 image was not created after "
                f"{RETRY_ATTEMPTS} attempts. "
                f"Last error: {last_error}"
            )

        record[
            "status"
        ] = "success"

        record[
            "file_size_bytes"
        ] = output_file.stat().st_size

        if CREATE_PREVIEWS:

            create_sentinel1_preview(
                output_file,
                preview_file,
            )

        return record

    except Exception as error:

        record[
            "status"
        ] = "failed"

        record[
            "error"
        ] = str(
            error
        )

        print(
            f"\nFailed: {site_id}, {group}, {period}"
        )

        print(
            error
        )

        return record


# =============================================================================
# 21. Test one treatment site
# =============================================================================

if TEST_DOWNLOAD_FIRST:

    test_site = (
        treatment_sites
        .iloc[0]
    )

    print(
        "\nTesting Sentinel-1 download for:"
    )

    print(
        test_site[
            "site_id"
        ]
    )

    test_record = download_site_image(
        test_site,
        "before",
    )

    print(
        "\nTest result:"
    )

    print(
        test_record
    )

    if test_record[
        "status"
    ] not in {
        "success",
        "existing",
    }:

        raise RuntimeError(
            "The test Sentinel-1 download did not succeed. "
            "Resolve the error before running the complete loop."
        )


# =============================================================================
# 22. Download Sentinel-1 images for all sites and periods
# =============================================================================

download_records = []

total_downloads = (
    len(
        all_sites
    )
    *
    len(
        IMAGE_PERIODS
    )
)

progress_bar = tqdm(
    total=total_downloads,
    desc="Downloading Sentinel-1 image chips",
)

for _, site_row in all_sites.iterrows():

    for period in [
        "before",
        "after",
    ]:

        record = download_site_image(
            site_row,
            period,
        )

        download_records.append(
            record
        )

        progress_bar.update(
            1
        )

progress_bar.close()


# =============================================================================
# 23. Save Sentinel-1 image inventory
# =============================================================================

sentinel1_inventory = pd.DataFrame(
    download_records
)

SENTINEL1_INVENTORY_FILE = (
    SENTINEL1_DIR /
    "sentinel1_image_inventory.csv"
)

sentinel1_inventory.to_csv(
    SENTINEL1_INVENTORY_FILE,
    index=False,
)

print(
    "\nSentinel-1 inventory saved to:"
)

print(
    SENTINEL1_INVENTORY_FILE
)


# =============================================================================
# 24. Display status counts
# =============================================================================

print(
    "\nDownload status counts:"
)

print(
    sentinel1_inventory[
        "status"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nStatus by group and period:"
)

print(
    sentinel1_inventory
    .groupby(
        [
            "group",
            "period",
            "status",
        ]
    )
    .size()
)


# =============================================================================
# 25. Check before-after completeness
# =============================================================================

successful_statuses = [
    "success",
    "existing",
]

successful_inventory = (
    sentinel1_inventory
    .loc[
        sentinel1_inventory[
            "status"
        ].isin(
            successful_statuses
        )
    ]
    .copy()
)

site_completeness = (
    successful_inventory
    .pivot_table(
        index=[
            "site_id",
            "pair_id",
            "group",
        ],
        columns="period",
        values="image_path",
        aggfunc="first",
    )
    .reset_index()
)

if "before" not in site_completeness.columns:

    site_completeness[
        "before"
    ] = np.nan

if "after" not in site_completeness.columns:

    site_completeness[
        "after"
    ] = np.nan

site_completeness[
    "has_before"
] = site_completeness[
    "before"
].notna()

site_completeness[
    "has_after"
] = site_completeness[
    "after"
].notna()

site_completeness[
    "complete_before_after"
] = (
    site_completeness[
        "has_before"
    ]
    &
    site_completeness[
        "has_after"
    ]
)

SITE_COMPLETENESS_FILE = (
    SENTINEL1_DIR /
    "sentinel1_site_completeness.csv"
)

site_completeness.to_csv(
    SITE_COMPLETENESS_FILE,
    index=False,
)

print(
    "\nSite before-after completeness:"
)

print(
    site_completeness[
        "complete_before_after"
    ].value_counts()
)


# =============================================================================
# 26. Check complete treatment-counterfactual pairs
# =============================================================================

site_period_status = (
    successful_inventory
    .assign(
        available=True
    )
    .pivot_table(
        index="pair_id",
        columns=[
            "group",
            "period",
        ],
        values="available",
        aggfunc="max",
        fill_value=False,
    )
)

required_columns = [
    (
        "treatment",
        "before",
    ),
    (
        "treatment",
        "after",
    ),
    (
        "counterfactual",
        "before",
    ),
    (
        "counterfactual",
        "after",
    ),
]

for column in required_columns:

    if column not in site_period_status.columns:

        site_period_status[
            column
        ] = False

site_period_status[
    "complete_pair"
] = (
    site_period_status[
        required_columns
    ]
    .all(
        axis=1
    )
)

matched_pair_completeness = (
    site_period_status
    .reset_index()
)

PAIR_COMPLETENESS_FILE = (
    SENTINEL1_DIR /
    "sentinel1_matched_pair_completeness.csv"
)

matched_pair_completeness.to_csv(
    PAIR_COMPLETENESS_FILE,
    index=False,
)

print(
    "\nMatched-pair completeness:"
)

print(
    matched_pair_completeness[
        "complete_pair"
    ].value_counts()
)


# =============================================================================
# 27. Validate downloaded GeoTIFFs
# =============================================================================

validation_records = []

for _, record in successful_inventory.iterrows():

    image_file = Path(
        record[
            "image_path"
        ]
    )

    validation_record = {
        "site_id": record["site_id"],
        "pair_id": record["pair_id"],
        "group": record["group"],
        "period": record["period"],
        "image_path": str(
            image_file
        ),
        "exists": image_file.exists(),
        "readable": False,
        "band_count": np.nan,
        "width": np.nan,
        "height": np.nan,
        "crs": None,
        "resolution_x": np.nan,
        "resolution_y": np.nan,
        "valid_pixel_fraction": np.nan,
        "vv_mean_db": np.nan,
        "vh_mean_db": np.nan,
        "vv_minus_vh_mean_db": np.nan,
        "error": None,
    }

    try:

        with rasterio.open(
            image_file
        ) as source:

            data = source.read(
                masked=True
            )

            validation_record[
                "readable"
            ] = True

            validation_record[
                "band_count"
            ] = source.count

            validation_record[
                "width"
            ] = source.width

            validation_record[
                "height"
            ] = source.height

            validation_record[
                "crs"
            ] = str(
                source.crs
            )

            validation_record[
                "resolution_x"
            ] = source.res[
                0
            ]

            validation_record[
                "resolution_y"
            ] = source.res[
                1
            ]

            if data.size > 0:

                valid_mask = (
                    ~np.ma.getmaskarray(
                        data
                    )
                )

                validation_record[
                    "valid_pixel_fraction"
                ] = float(
                    valid_mask.mean()
                )

            if source.count >= 3:

                validation_record[
                    "vv_mean_db"
                ] = float(
                    data[0].mean()
                )

                validation_record[
                    "vh_mean_db"
                ] = float(
                    data[1].mean()
                )

                validation_record[
                    "vv_minus_vh_mean_db"
                ] = float(
                    data[2].mean()
                )

    except Exception as error:

        validation_record[
            "error"
        ] = str(
            error
        )

    validation_records.append(
        validation_record
    )

validation_table = pd.DataFrame(
    validation_records
)

VALIDATION_FILE = (
    SENTINEL1_DIR /
    "sentinel1_image_validation.csv"
)

validation_table.to_csv(
    VALIDATION_FILE,
    index=False,
)

print(
    "\nSentinel-1 validation saved to:"
)

print(
    VALIDATION_FILE
)

print(
    "\nReadable image counts:"
)

print(
    validation_table[
        "readable"
    ].value_counts()
)

print(
    "\nBand counts:"
)

print(
    validation_table[
        "band_count"
    ].value_counts(
        dropna=False
    )
)


# =============================================================================
# 28. Create analysis-ready inventory
# =============================================================================

complete_site_ids = set(
    site_completeness.loc[
        site_completeness[
            "complete_before_after"
        ],
        "site_id",
    ]
)

analysis_ready_inventory = (
    successful_inventory
    .loc[
        successful_inventory[
            "site_id"
        ].isin(
            complete_site_ids
        )
    ]
    .copy()
)

ANALYSIS_READY_INVENTORY_FILE = (
    SENTINEL1_DIR /
    "sentinel1_analysis_ready_inventory.csv"
)

analysis_ready_inventory.to_csv(
    ANALYSIS_READY_INVENTORY_FILE,
    index=False,
)

print(
    "\nAnalysis-ready Sentinel-1 inventory saved to:"
)

print(
    ANALYSIS_READY_INVENTORY_FILE
)


# =============================================================================
# 29. Summarize output folders
# =============================================================================

def count_tiff_files(
    folder,
):
    """
    Count TIFF files directly inside a folder.
    """

    return len(
        list(
            Path(
                folder
            ).glob(
                "*.tif"
            )
        )
    )


folder_summary = pd.DataFrame(
    [
        {
            "group": "treatment",
            "period": "before",
            "folder": str(
                S1_TREATMENT_BEFORE_DIR
            ),
            "tiff_count": count_tiff_files(
                S1_TREATMENT_BEFORE_DIR
            ),
        },
        {
            "group": "treatment",
            "period": "after",
            "folder": str(
                S1_TREATMENT_AFTER_DIR
            ),
            "tiff_count": count_tiff_files(
                S1_TREATMENT_AFTER_DIR
            ),
        },
        {
            "group": "counterfactual",
            "period": "before",
            "folder": str(
                S1_COUNTERFACTUAL_BEFORE_DIR
            ),
            "tiff_count": count_tiff_files(
                S1_COUNTERFACTUAL_BEFORE_DIR
            ),
        },
        {
            "group": "counterfactual",
            "period": "after",
            "folder": str(
                S1_COUNTERFACTUAL_AFTER_DIR
            ),
            "tiff_count": count_tiff_files(
                S1_COUNTERFACTUAL_AFTER_DIR
            ),
        },
    ]
)

FOLDER_SUMMARY_FILE = (
    SENTINEL1_DIR /
    "sentinel1_folder_summary.csv"
)

folder_summary.to_csv(
    FOLDER_SUMMARY_FILE,
    index=False,
)

print(
    "\nFinal Sentinel-1 folder summary:"
)

print(
    folder_summary
)


# =============================================================================
# 30. Final validation
# =============================================================================

print(
    "\n"
    + "=" * 78
)

print(
    "SENTINEL-1 IMAGE DOWNLOAD SUMMARY"
)

print(
    "=" * 78
)

print(
    f"Selected orbit pass: "
    f"{ORBIT_PASS}"
)

print(
    f"Total requested images: "
    f"{len(sentinel1_inventory):,}"
)

print(
    f"Successful or existing images: "
    f"{len(successful_inventory):,}"
)

print(
    f"Failed images: "
    f"{(
        sentinel1_inventory['status']
        == 'failed'
    ).sum():,}"
)

print(
    f"No-scene images: "
    f"{(
        sentinel1_inventory['status']
        == 'no_scenes'
    ).sum():,}"
)

print(
    f"Sites with complete before-after images: "
    f"{site_completeness['complete_before_after'].sum():,}"
)

print(
    f"Complete treatment-counterfactual pairs: "
    f"{matched_pair_completeness['complete_pair'].sum():,}"
)

print(
    "\nSentinel-1 treatment before folder:"
)

print(
    S1_TREATMENT_BEFORE_DIR
)

print(
    "\nSentinel-1 treatment after folder:"
)

print(
    S1_TREATMENT_AFTER_DIR
)

print(
    "\nSentinel-1 counterfactual before folder:"
)

print(
    S1_COUNTERFACTUAL_BEFORE_DIR
)

print(
    "\nSentinel-1 counterfactual after folder:"
)

print(
    S1_COUNTERFACTUAL_AFTER_DIR
)

print(
    "\nNotebook completed."
)

Packages loaded successfully.
Sentinel-1 output directory:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel1
Required site files found.
Earth Engine initialized using existing credentials.
Image periods:
before 2024-08-15 2024-09-23
after 2024-10-01 2024-10-31
Treatment sites: 26
Counterfactual sites: 260
Total sites: 286

Sentinel-1 orbit diagnostics:
   orbit_pass  period  start_date    end_date  scene_count
0   ASCENDING  before  2024-08-15  2024-09-23            7
1   ASCENDING   after  2024-10-01  2024-10-31            4
2  DESCENDING  before  2024-08-15  2024-09-23            0
3  DESCENDING   after  2024-10-01  2024-10-31            0

Selected orbit pass:
ASCENDING

Available Sentinel-1 bands:
['VV', 'VH', 'angle']

Example Sentinel-1 image properties:
{'instrumentMode': 'IW', 'orbitProperties_pass': 'ASCENDING', 'relativeOrbitNumber_start': 48, 'resolution_meters': 10, 'transmitterReceiverPolarisation': ['VV', 'VH']}

Testing Sentinel-1 down


Generating URL ...
Please wait ...
Data downloaded to /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel1/counterfactual/before/counterfactual_0001_01_before_sentinel1.tif

Generating URL ...
Please wait ...
Data downloaded to /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel1/counterfactual/after/counterfactual_0001_01_after_sentinel1.tif

Generating URL ...
Please wait ...
Data downloaded to /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel1/counterfactual/before/counterfactual_0001_02_before_sentinel1.tif

Generating URL ...
Please wait ...
Data downloaded to /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel1/counterfactual/after/counterfactual_0001_02_after_sentinel1.tif

Generating URL ...
Please wait ...
Data downloaded to /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/sentinel1/counterfactual/before/counterfactua